# Data Ingestion Phase

Thin runner for bronze + silver. Pipeline logic lives in `src/`.

```bash
python -m src.jobs.run_download
python -m src.jobs.run_bronze
python -m src.jobs.run_silver
```

## 0. Download raw datasets

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.spark import ensure_runtime

ensure_runtime()

from src.bronze.download import download_raw

download_raw()

/Users/juozas/Documents/Projects/kth/id2221_labs/id2221-labs/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
Retrieving folder contents


Processing file 1NGT8RR-NtBf-4xxII4pfwBanE_BgBvoW air_quality.zip
Processing file 1-gO-MhXTIPbHRkyJQo8nTxly72gT3r9t taxi_zone_lookup.csv
Processing file 1q-Lw24XFqJ42XSJRuUV_ced3aJ6kKQ2M weather.csv
Processing file 17v0eFEontYEKtGoqyB0v9rj_BEDc7snA yellow_tripdata_2024-01.parquet
Processing file 1N-dRuGdd_lOYGAbdbgJJWMyIsV_lz057 yellow_tripdata_2024-02.parquet
Processing file 1oUxC0cLWqOatddyT8aB06VvFFZY0-lu2 yellow_tripdata_2024-03.parquet


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1NGT8RR-NtBf-4xxII4pfwBanE_BgBvoW
From (redirected): https://drive.google.com/uc?id=1NGT8RR-NtBf-4xxII4pfwBanE_BgBvoW&confirm=t&uuid=ead4326d-8dfd-40a4-9f2b-a92d51526d38
To: /Users/juozas/Documents/Projects/kth/id2221_labs/id2221-labs/data/drive_download/air_quality.zip
100%|██████████| 66.3M/66.3M [00:01<00:00, 57.5MB/s]
Downloading...
From: https://drive.google.com/uc?id=1-gO-MhXTIPbHRkyJQo8nTxly72gT3r9t
To: /Users/juozas/Documents/Projects/kth/id2221_labs/id2221-labs/data/drive_download/taxi_zone_lookup.csv
100%|██████████| 12.3k/12.3k [00:00<00:00, 5.44MB/s]
Downloading...
From: https://drive.google.com/uc?id=1q-Lw24XFqJ42XSJRuUV_ced3aJ6kKQ2M
To: /Users/juozas/Documents/Projects/kth/id2221_labs/id2221-labs/data/drive_download/weather.csv
100%|██████████| 1.07M/1.07M [00:00<00:00, 25.1MB/s]
Downloading...
From: https:

unzip data/drive_download/air_quality.zip -> data/raw/air_quality
copy  yellow_tripdata_2024-03.parquet -> data/raw/taxi_trips/yellow
copy  taxi_zone_lookup.csv -> data/raw/taxi_zones
copy  weather.csv -> data/raw/weather
copy  yellow_tripdata_2024-02.parquet -> data/raw/taxi_trips/yellow
copy  yellow_tripdata_2024-01.parquet -> data/raw/taxi_trips/yellow


PosixPath('/Users/juozas/Documents/Projects/kth/id2221_labs/id2221-labs/data/raw')

## 1. Bronze ingest (schema validation)

In [2]:
from src.spark import create_spark, ensure_runtime

ensure_runtime()

from src.lake import BRONZE, show_delta
from src.bronze.ingest import ingest_all

spark = create_spark("ingestion-notebook")
ingest_all(spark)

for table in ["taxi_zones", "weather", "air_quality", "taxi_trips"]:
    show_delta(spark, BRONZE / table)

:: loading settings :: url = jar:file:/Users/juozas/Documents/Projects/kth/id2221_labs/id2221-labs/.venv/lib/python3.9/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/juozas/.ivy2/cache
The jars for the packages stored in: /Users/juozas/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-997aefaf-8890-4718-b077-16e55d93de0d;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.1 in central
	found io.delta#delta-storage;3.2.1 in central
	found org.antlr#antlr4-runtime;4.9.3 in local-m2-cache
:: resolution report :: resolve 111ms :: artifacts dl 4ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.1 from central in [default]
	io.delta#delta-storage;3.2.1 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from local-m2-cache in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default   

[air_quality] schema ok=True


26/09/24 15:45:40 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


[bronze/air_quality] written
[taxi_trips/yellow] schema ok=True


[bronze/taxi_trips] written
[taxi_zones] schema ok=True
[bronze/taxi_zones] written
[weather] schema ok=True


[bronze/weather] written
taxi_zones: 265 rows @ data/lake/bronze/taxi_zones
+----------+-------+-----------------------+------------+--------------------------+
|LocationID|Borough|Zone                   |service_zone|_ingested_at              |
+----------+-------+-----------------------+------------+--------------------------+
|1         |EWR    |Newark Airport         |EWR         |2026-09-24 13:46:06.598616|
|2         |Queens |Jamaica Bay            |Boro Zone   |2026-09-24 13:46:06.598616|
|3         |Bronx  |Allerton/Pelham Gardens|Boro Zone   |2026-09-24 13:46:06.598616|
+----------+-------+-----------------------+------------+--------------------------+
only showing top 3 rows

weather: 8784 rows @ data/lake/bronze/weather
+----+-----+---+----+----+-----------+----+-----------+----+-----------+----+-----------+----+-----------+----+-----------+----+-----------+------+-----------+----+-----------+----+-----------+----------------+--------------------------+
|year|month|day|hour

## 2. Silver promote (row DQ + rejects)

In [3]:
from src.lake import SILVER
from src.silver.promote import promote_all

promote_all(spark)

print("Silver Delta tables")
for table in ["taxi_zones", "weather", "air_quality", "taxi_trips"]:
    show_delta(spark, SILVER / table)

[silver/air_quality] kept=112,838  rejected=4,600


[silver/taxi_trips] kept=9,417,383  rejected=137,395
[silver/taxi_zones] kept=265  rejected=0


[silver/weather] kept=8,784  rejected=0
Silver Delta tables
taxi_zones: 265 rows @ data/lake/silver/taxi_zones
+-----------+-------+-----------------------+------------+
|location_id|borough|zone                   |service_zone|
+-----------+-------+-----------------------+------------+
|1          |EWR    |Newark Airport         |EWR         |
|2          |Queens |Jamaica Bay            |Boro Zone   |
|3          |Bronx  |Allerton/Pelham Gardens|Boro Zone   |
+-----------+-------+-----------------------+------------+
only showing top 3 rows

weather: 8784 rows @ data/lake/silver/weather
+-----------+-------------------+-------------+-----------------+----------------+----------------+
|station_id |obs_timestamp      |temperature_c|wind_speed_ms    |observation_date|observation_hour|
+-----------+-------------------+-------------+-----------------+----------------+----------------+
|72505394728|2024-01-02 00:00:00|6.1          |4.111111111111112|2024-01-02      |0               |
|7250

In [4]:
spark.stop()